In [13]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from scipy.stats import ks_2samp
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier
from scipy.stats import ks_2samp
import h2o
from h2o.automl import H2OAutoML
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


file_path = os.path.join('..', 'Data', 'processed', 'Data.pkl')
Data = pd.read_pickle(file_path)

In [2]:
X = Data.drop(columns=['target'])
y = Data['target']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

In [3]:
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'max_depth': [-1, 5, 10],
    'n_estimators': [100, 200, 500]
}

# Crear el modelo base
lgb_estimator = lgb.LGBMClassifier(objective='binary', metric='auc', boosting_type='gbdt', random_state=42)

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=lgb_estimator,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=-1
)

# Entrenar GridSearchCV
grid_search.fit(X_train, y_train)

# Mejores parámetros y rendimiento
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor ROC AUC en validación:", grid_search.best_score_)

# Usar el mejor modelo
best_lgb_model = grid_search.best_estimator_

# Evaluar en el conjunto de prueba
y_pred_proba_best_lgb = best_lgb_model.predict_proba(X_test)[:, 1]
y_pred_best_lgb = (y_pred_proba_best_lgb > 0.5).astype(int)

roc_auc_best_lgb = roc_auc_score(y_test, y_pred_proba_best_lgb)
accuracy_best_lgb = accuracy_score(y_test, y_pred_best_lgb)
ks_stat_best_lgb = ks_2samp(y_pred_proba_best_lgb[y_test == 1], y_pred_proba_best_lgb[y_test == 0]).statistic

print(f"LightGBM Optimizado - ROC AUC: {roc_auc_best_lgb:.4f}, Accuracy: {accuracy_best_lgb:.4f}, KS: {ks_stat_best_lgb:.4f}")

Fitting 3 folds for each of 81 candidates, totalling 243 fits
[LightGBM] [Info] Number of positive: 790, number of negative: 26499
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007916 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10582
[LightGBM] [Info] Number of data points in the train set: 27289, number of used features: 79
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.028949 -> initscore=-3.512829
[LightGBM] [Info] Start training from score -3.512829
Mejores parámetros: {'learning_rate': 0.01, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Mejor ROC AUC en validación: 0.6335228064899118
LightGBM Optimizado - ROC AUC: 0.6669, Accuracy: 0.9711, KS: 0.2786


In [4]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=True)]

model_lgb = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
    callbacks=callbacks
)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[32]	valid_0's auc: 0.641267


In [5]:
y_pred_proba = model_lgb.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)
ks_stat = ks_2samp(y_pred_proba[y_test == 1], y_pred_proba[y_test == 0]).statistic

print(f"ROC AUC: {roc_auc:.4f}, Accuracy: {accuracy:.4f}, KS: {ks_stat:.4f}")

ROC AUC: 0.6762, Accuracy: 0.9711, KS: 0.2876


-------------

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

y_train = y_train.values.ravel() if isinstance(y_train, pd.Series) else y_train.ravel()
y_val = y_val.values.ravel() if isinstance(y_val, pd.Series) else y_val.ravel()
y_test = y_test.values.ravel() if isinstance(y_test, pd.Series) else y_test.ravel()


unique_classes = np.unique(y_train)
print("Clases únicas en y_train:", unique_classes)
print("Tipo de datos en y_train:", y_train.dtype)

class_weights = compute_class_weight(
    'balanced', 
    classes=unique_classes,  
    y=y_train
)


class_weights_dict = {int(cls): weight for cls, weight in zip(unique_classes, class_weights)}
print("Pesos de las clases ajustados:", class_weights_dict)


model_nn = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),  
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])


model_nn.compile(
    optimizer=Adam(learning_rate=0.001), 
    loss='binary_crossentropy', 
    metrics=['AUC']
)

history = model_nn.fit(
    X_train_scaled, 
    y_train, 
    validation_data=(X_val_scaled, y_val), 
    epochs=50, 
    batch_size=32, 
    class_weight=class_weights_dict, 
    verbose=1
)

Clases únicas en y_train: [0. 1.]
Tipo de datos en y_train: float64
Pesos de las clases ajustados: {0: 0.5149062228763349, 1: 17.27151898734177}
Epoch 1/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - AUC: 0.5413 - loss: 0.7415 - val_AUC: 0.6335 - val_loss: 0.6955
Epoch 2/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - AUC: 0.6276 - loss: 0.6801 - val_AUC: 0.6355 - val_loss: 0.6352
Epoch 3/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - AUC: 0.6465 - loss: 0.6706 - val_AUC: 0.6566 - val_loss: 0.5885
Epoch 4/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - AUC: 0.6840 - loss: 0.6320 - val_AUC: 0.6465 - val_loss: 0.6334
Epoch 5/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - AUC: 0.6790 - loss: 0.6360 - val_AUC: 0.6480 - val_loss: 0.6109
Epoch 6/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - AUC: 0.6892 - loss: 0.6352 - val_AUC: 0.6578 - val_loss: 0.6021
Epoch 7/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - AUC: 0.7051 - loss: 0.6247 - val_AUC: 0.6394 - val_loss: 0.6311
Epoch 8/50
853/853 ━━━━━━━━━━

In [12]:
y_pred_proba_nn = model_nn.predict(X_test_scaled).flatten()
y_pred_nn = (y_pred_proba_nn > 0.5).astype(int)

roc_auc_nn = roc_auc_score(y_test, y_pred_proba_nn)
accuracy_nn = accuracy_score(y_test, y_pred_nn)
ks_stat_nn = ks_2samp(y_pred_proba_nn[y_test == 1], y_pred_proba_nn[y_test == 0]).statistic

print(f"ROC AUC (NN): {roc_auc_nn:.4f}, Accuracy (NN): {accuracy_nn:.4f}, KS (NN): {ks_stat_nn:.4f}")

183/183 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
ROC AUC (NN): 0.6321, Accuracy (NN): 0.7196, KS (NN): 0.2399


In [14]:
model_catboost = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,  
    depth=8,          
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',  
    verbose=100,
    random_state=42
)

model_catboost.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100  
)

y_pred_proba_catboost = model_catboost.predict_proba(X_test)[:, 1]
y_pred_catboost = (y_pred_proba_catboost > 0.5).astype(int)

roc_auc_catboost = roc_auc_score(y_test, y_pred_proba_catboost)
accuracy_catboost = accuracy_score(y_test, y_pred_catboost)
ks_stat_catboost = ks_2samp(y_pred_proba_catboost[y_test == 1], y_pred_proba_catboost[y_test == 0]).statistic
precision_catboost = precision_score(y_test, y_pred_catboost)
recall_catboost = recall_score(y_test, y_pred_catboost)
f1_catboost = f1_score(y_test, y_pred_catboost)

print(f"CatBoost - ROC AUC: {roc_auc_catboost:.4f}, Accuracy: {accuracy_catboost:.4f}, KS: {ks_stat_catboost:.4f}")
print(f"CatBoost - Precision: {precision_catboost:.4f}, Recall: {recall_catboost:.4f}, F1: {f1_catboost:.4f}")


0:	test: 0.5561547	best: 0.5561547 (0)	total: 260ms	remaining: 4m 19s
100:	test: 0.6272023	best: 0.6308866 (37)	total: 7.14s	remaining: 1m 3s
200:	test: 0.6252142	best: 0.6362046 (115)	total: 11.8s	remaining: 46.7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6362045989
bestIteration = 115

Shrink model to first 116 iterations.
CatBoost - ROC AUC: 0.6177, Accuracy: 0.8365, KS: 0.1863
CatBoost - Precision: 0.0523, Recall: 0.2722, F1: 0.0878


In [18]:
print("Tipo de X_train:", type(X_train))
print("Tipo de y_train:", type(y_train))
print("Tipo de X_val:", type(X_val))
print("Tipo de y_val:", type(y_val))
print("Tipo de X_test:", type(X_test))
print("Tipo de y_test:", type(y_test))

Tipo de X_train: <class 'pandas.core.frame.DataFrame'>
Tipo de y_train: <class 'pandas.core.series.Series'>
Tipo de X_val: <class 'pandas.core.frame.DataFrame'>
Tipo de y_val: <class 'pandas.core.series.Series'>
Tipo de X_test: <class 'pandas.core.frame.DataFrame'>
Tipo de y_test: <class 'pandas.core.series.Series'>


In [19]:
h2o.init()

y_train = pd.Series(y_train, name='target').astype('category')
y_val = pd.Series(y_val, name='target').astype('category')
y_test = pd.Series(y_test, name='target').astype('category')

train_h2o = h2o.H2OFrame(pd.concat([X_train, y_train], axis=1))
val_h2o = h2o.H2OFrame(pd.concat([X_val, y_val], axis=1))
test_h2o = h2o.H2OFrame(pd.concat([X_test, y_test], axis=1))

features = list(X_train.columns)
target = 'target'

aml = H2OAutoML(
    max_models=10,
    seed=42,
    stopping_metric='AUC',
    sort_metric='AUC',
    max_runtime_secs=600  # Límite de tiempo de 10 minutos
)
aml.train(x=features, y=target, training_frame=train_h2o, validation_frame=val_h2o)

best_model = aml.leader

if best_model is not None:
    predictions = best_model.predict(test_h2o)
    y_pred_proba_automl = predictions.as_data_frame()['p1'].values
    y_pred_automl = (y_pred_proba_automl > 0.5).astype(int)

    roc_auc_automl = roc_auc_score(y_test, y_pred_proba_automl)
    accuracy_automl = accuracy_score(y_test, y_pred_automl)
    ks_stat_automl = ks_2samp(y_pred_proba_automl[y_test == 1], y_pred_proba_automl[y_test == 0]).statistic

    print(f"AutoML (H2O) - ROC AUC: {roc_auc_automl:.4f}, Accuracy: {accuracy_automl:.4f}, KS: {ks_stat_automl:.4f}")
else:
    print("No se pudo entrenar ningún modelo.")

h2o.shutdown()

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,32 secs
H2O_cluster_timezone:,America/Mexico_City
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,4 months and 25 days
H2O_cluster_name:,H2O_from_python_Usuario_62egvf
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.862 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%

13:34:23.418: User specified a validation frame with cross-validation still enabled. Please note that the models will still be validated using cross-validation only, the validation frame will be used to provide purely informative validation metrics on the trained models.
13:34:23.420: AutoML: XGBoost is not available; skipping it.
13:34:23.431: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.
13:34:23.431: _validation_frame param, Test/Validation

C:\Users\Usuario\AppData\Local\Temp\ipykernel_18956\1993978284.py:38: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown()


H2O session _sid_b767 closed.


## Probabilistic Ensemble 

In [22]:
y_pred_proba_ensemble = (y_pred_proba + y_pred_proba_nn + y_pred_proba_catboost) / 3
y_pred_ensemble = (y_pred_proba_ensemble > 0.5).astype(int)

roc_auc_ensemble = roc_auc_score(y_test, y_pred_proba_ensemble)
accuracy_ensemble = accuracy_score(y_test, y_pred_ensemble)
ks_stat_ensemble = ks_2samp(y_pred_proba_ensemble[y_test == 1], y_pred_proba_ensemble[y_test == 0]).statistic

print(f"Ensemble - ROC AUC: {roc_auc_ensemble:.4f}, Accuracy: {accuracy_ensemble:.4f}, KS: {ks_stat_ensemble:.4f}")

Ensemble - ROC AUC: 0.6472, Accuracy: 0.9627, KS: 0.2438


### Stacking Ensemble

In [24]:
model_lgb.fit(X_train, y_train)
model_nn.fit(X_train_scaled, y_train, epochs=50, batch_size=32, class_weight=class_weights_dict, verbose=0)
model_catboost.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100, verbose=0)


X_train_meta = np.column_stack([
    model_lgb.predict_proba(X_train)[:, 1],
    model_nn.predict(X_train_scaled).flatten(),
    model_catboost.predict_proba(X_train)[:, 1]
])

X_val_meta = np.column_stack([
    model_lgb.predict_proba(X_val)[:, 1],
    model_nn.predict(X_val_scaled).flatten(),
    model_catboost.predict_proba(X_val)[:, 1]
])

X_test_meta = np.column_stack([
    model_lgb.predict_proba(X_test)[:, 1],
    model_nn.predict(X_test_scaled).flatten(),
    model_catboost.predict_proba(X_test)[:, 1]
])

meta_model = LogisticRegression()
meta_model.fit(X_train_meta, y_train)

y_pred_proba_stack = meta_model.predict_proba(X_test_meta)[:, 1]
y_pred_stack = (y_pred_proba_stack > 0.5).astype(int)

roc_auc_stack = roc_auc_score(y_test, y_pred_proba_stack)
accuracy_stack = accuracy_score(y_test, y_pred_stack)
ks_stat_stack = ks_2samp(y_pred_proba_stack[y_test == 1], y_pred_proba_stack[y_test == 0]).statistic

print(f"Stacking - ROC AUC: {roc_auc_stack:.4f}, Accuracy: {accuracy_stack:.4f}, KS: {ks_stat_stack:.4f}")

AttributeError: 'Booster' object has no attribute 'fit'